# 03 – Multifractality Diagnostics

MF-DFA, moment scaling, and wavelet leaders.
Reference: Pontiggia (2025) §3.6.2 and §4.3.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from src.diagnostics.mfdfa import hurst_exponents, shuffle_control
from src.diagnostics.moment_scaling import scaling_exponents
from src.diagnostics.wavelet_leaders import wavelet_scaling_exponents
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

## Load series

In [ ]:
fpath = Path('../data/processed/rv_binance_BTC_USDT_1m_2021.parquet')
rv = pd.read_parquet(fpath)['rv'].dropna().to_numpy() if fpath.exists() else np.abs(np.random.default_rng(1).standard_normal(5000))
print(f'N={len(rv)}')

## MF-DFA H(q) spectrum with shuffle control

In [ ]:
q = np.linspace(-4, 4, 17)
mfc = cfg['mfdfa']
kw = dict(min_scale=mfc['min_scale'], max_scale_pct=mfc['max_scale_pct'])
orig = hurst_exponents(rv, q, **kw)
shuf = shuffle_control(rv, q, n_shuffles=mfc['n_shuffles'], **kw)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q, orig['H'], 'b-o', ms=5, label='Original')
ax.plot(q, shuf['H_mean'], 'r--s', ms=5, label='Shuffled')
ax.fill_between(q, shuf['H_mean']-shuf['H_std'], shuf['H_mean']+shuf['H_std'], alpha=0.2, color='r')
ax.set_xlabel('q'); ax.set_ylabel('H(q)'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
print(f"Width orig: {np.nanmax(orig['H'])-np.nanmin(orig['H']):.4f}")
print(f"Width shuf: {np.nanmax(shuf['H_mean'])-np.nanmin(shuf['H_mean']):.4f}")

## zeta_q curves (moment scaling and wavelet leaders)

In [ ]:
ms = scaling_exponents(rv, q, tau_min=cfg['moment_scaling']['tau_min'], tau_max_pct=cfg['moment_scaling']['tau_max_pct'])
wl = wavelet_scaling_exponents(rv, q, wavelet=cfg['wavelet']['wavelet'], min_j=cfg['wavelet']['min_scale_j'], max_j=cfg['wavelet']['max_scale_j'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, r, t in zip(axes, [ms, wl], ['Moment scaling', 'Wavelet leaders']):
    ax.plot(q, r['zeta_q'], 'b-o', ms=5)
    ax.plot(q, 0.5*q, 'k--', alpha=0.5, label='Linear H=0.5')
    ax.set_xlabel('q'); ax.set_ylabel('zeta_q'); ax.set_title(t); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()